# Trust-dimension agent search and evidence collection

This is the only notebook that calls remote services for the trust-model
comparison. It runs a purpose-specific agent search for every dimension in
`trust_score_implementation_notes.md`, records which search feeds which
comparison section, and stores complete captures in two stable JSON bundles:

- `trust_model_search_bundle.json` — search plans, raw search responses, and candidate summaries.
- `trust_model_evidence_bundle.json` — detail, document, validation, DNS, and ANS captures.

Re-running the notebook replaces those two current-snapshot files. Copy them to
a date-stamped archive first when a historical comparison is required.

## Setup and dimension map

Search rank is candidate discovery, not trust evidence. The queries are tuned
to find useful subjects for each dimension; the `evidence_boundary` text states
what the currently available APIs can actually support.

In [2]:
import hashlib
import json
import ssl
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen

try:
    import EXPERIMENTS.notebooks.utils as experiment_utils
except ModuleNotFoundError:
    import utils as experiment_utils

agentcensus_get_json = experiment_utils.agentcensus_get_json
agentcensus_load_api_key = experiment_utils.agentcensus_load_api_key
a2a_registry_request_json = experiment_utils.a2a_registry_request_json

AGENTCENSUS_BEARER_TOKEN = agentcensus_load_api_key(required=True)
NOTEBOOK_DIR = Path(experiment_utils.__file__).resolve().parent
OUTPUT_DIR = NOTEBOOK_DIR.parent / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEARCH_BUNDLE_PATH = OUTPUT_DIR / "trust_model_search_bundle.json"
EVIDENCE_BUNDLE_PATH = OUTPUT_DIR / "trust_model_evidence_bundle.json"
ANS_DNS_RESOLVER_URL = "https://dns.google/resolve"
ANS_TLS_CONTEXT = experiment_utils.agentcensus_trusted_tls_context()

DIMENSION_PLANS = {
    "identity_control": {
        "title": "Identity and control",
        "query": "Find an A2A agent with verifiable operator identity and domain control",
        "evidence_boundary": "Search finds candidates only; control requires separately verified bindings.",
    },
    "integrity": {
        "title": "Integrity",
        "query": "Find an A2A agent with signed discovery records and stable versioned metadata",
        "evidence_boundary": "Search claims are hypotheses; compare signatures and live values with captured baselines.",
    },
    "behavioral_reliability": {
        "title": "Behavioral reliability",
        "query": "Find an A2A agent with public reliability history and repeatable task outcomes",
        "evidence_boundary": "The available public reads do not measure repeated task correctness over a window.",
    },
    "security_authorization": {
        "title": "Security and authorization",
        "query": "Find an A2A agent declaring scoped authentication and authorization controls",
        "evidence_boundary": "Declarations identify test targets; enforcement requires active authorization tests.",
    },
    "safety": {
        "title": "Safety",
        "query": "Find an A2A agent with independently evaluated safety behavior",
        "evidence_boundary": "Infrastructure scans do not establish safe agent behavior under controlled tests.",
    },
    "claim_accuracy": {
        "title": "Claim accuracy",
        "query": "Find an A2A agent whose published capabilities can be independently verified",
        "evidence_boundary": "Cross-source agreement shows consistency, not that capability claims are true.",
    },
}

# Selections are deliberately dimension-scoped even when two dimensions happen
# to use the same examples today. Future edits can change one without silently
# changing another section.
BASE_COMPARISON_AGENTS = [
    {
        "case": "shared", "output_label": "shared_council_of_ai",
        "name": "Council of AI — Measurement Agent", "domain": "councilof.ai",
        "agentcensus_agent_key": "ag_0257be5ab061",
        "a2a_registry_package": "ai.councilof.council_of_ai__measurement_agent",
        "a2a_manifest_url": "https://councilof.ai/.well-known/agent.json",
    },
    {
        "case": "agentcensus_only", "output_label": "only_selnoviktech",
        "name": "Selnoviktech Agent", "domain": "selnoviktech.com",
        "agentcensus_agent_key": "ag_050fb7a9ceea",
        "a2a_registry_package": None, "a2a_manifest_url": None,
    },
    {
        "case": "a2a_registry_only", "output_label": "missing_a2a402",
        "name": "A2A402 Agent Marketplace", "domain": "a2a402.market",
        "agentcensus_agent_key": None,
        "a2a_registry_package": "market.a2a402.a2a402_agent_origin_market",
        "a2a_manifest_url": "https://a2a402.market/.well-known/agent-card.json",
    },
]
DIMENSION_SELECTIONS = {
    "identity_control": [dict(item) for item in BASE_COMPARISON_AGENTS],
    "integrity": [dict(item) for item in BASE_COMPARISON_AGENTS],
    "behavioral_reliability": [],
    "security_authorization": [],
    "safety": [],
    "claim_accuracy": [],
}

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(path)

def canonical_digest(payload):
    body = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(body).hexdigest()

def record_json(payload, *, url=None, sha256=None, hash_scope="canonical-json"):
    return {
        "mediaType": "application/json",
        "url": url,
        "sha256": sha256 or canonical_digest(payload),
        "hashScope": hash_scope,
        "payload": payload,
    }

## Run one search per trust dimension

In [3]:
def summarize_candidates(agentcensus_response, registry_response, limit=5):
    census_rows = []
    census_data = agentcensus_response.get("data") or {}
    for result in census_data.get("results") or []:
        agent = result.get("agent") or {}
        census_rows.append({
            "name": agent.get("displayName"),
            "domain": agent.get("domain"),
            "agentKey": agent.get("agentKey"),
            "source": "AgentCensus",
        })
    registry_rows = []
    registry_data = registry_response.get("data") or {}
    for agent in registry_data.get("agents") or []:
        registry_rows.append({
            "name": agent.get("displayName"),
            "domain": agent.get("manifestUrl"),
            "packageName": agent.get("packageName"),
            "source": "A2A Registry",
        })
    return {"agentcensus": census_rows[:limit], "a2a_registry": registry_rows[:limit]}

search_dimensions = {}
for dimension_id, plan in DIMENSION_PLANS.items():
    agentcensus_response = agentcensus_get_json(
        "search",
        {"q": plan["query"], "mechanism": "a2a", "limit": 20},
        token=AGENTCENSUS_BEARER_TOKEN,
    )
    registry_response = a2a_registry_request_json(
        "GET", "/public/agents", params={"q": plan["query"], "page": 1},
        authenticated=False,
    )
    search_dimensions[dimension_id] = {
        **plan,
        "comparison_section": dimension_id,
        "raw": {
            "agentcensus": agentcensus_response,
            "a2a_registry": registry_response,
        },
        "candidates": summarize_candidates(agentcensus_response, registry_response),
    }

search_bundle = {
    "schemaVersion": "1.0",
    "capturedAt": datetime.now(timezone.utc).isoformat(),
    "dimensions": search_dimensions,
    "dimensionSelections": DIMENSION_SELECTIONS,
}
write_json(SEARCH_BUNDLE_PATH, search_bundle)

/Users/hanhanwu/Documents/Github/Agentopia/EXPERIMENTS/output/trust_model_search_bundle.json


## Review the section mapping

Candidate summaries keep this review compact. The complete responses remain in
the same search bundle under `dimensions.<id>.raw`.

In [4]:
for dimension_id, item in search_bundle["dimensions"].items():
    counts = {source: len(rows) for source, rows in item["candidates"].items()}
    print(dimension_id, "->", item["comparison_section"], counts, SEARCH_BUNDLE_PATH.name)

identity_control -> identity_control {'agentcensus': 5, 'a2a_registry': 5} trust_model_search_bundle.json
integrity -> integrity {'agentcensus': 5, 'a2a_registry': 5} trust_model_search_bundle.json
behavioral_reliability -> behavioral_reliability {'agentcensus': 3, 'a2a_registry': 5} trust_model_search_bundle.json
security_authorization -> security_authorization {'agentcensus': 5, 'a2a_registry': 5} trust_model_search_bundle.json
safety -> safety {'agentcensus': 5, 'a2a_registry': 5} trust_model_search_bundle.json
claim_accuracy -> claim_accuracy {'agentcensus': 5, 'a2a_registry': 5} trust_model_search_bundle.json


## Capture shared evidence once

Every unique response is stored once in `captures`, keyed by its former
descriptive filename so existing render helpers remain readable. Dimensions
select fields from this shared bundle in memory; they do not create derived
dimension-specific JSON files.

In [5]:
captures = {}

def add_capture(name, payload, *, url=None, raw_body=None):
    digest = hashlib.sha256(raw_body).hexdigest() if raw_body is not None else None
    scope = "response-bytes" if raw_body is not None else "canonical-json"
    captures[name] = record_json(payload, url=url, sha256=digest, hash_scope=scope)

selected_by_label = {}
for dimension_agents in DIMENSION_SELECTIONS.values():
    for agent in dimension_agents:
        selected_by_label[agent["output_label"]] = agent

agent_responses = {}
for agent in selected_by_label.values():
    agent_key = agent["agentcensus_agent_key"]
    if agent_key:
        response = agentcensus_get_json(f"agents/{agent_key}", token=AGENTCENSUS_BEARER_TOKEN)
        agent_responses[agent_key] = response
        add_capture(f"trust_model_comparison_agentcensus_{agent['output_label']}_agent.json", response)
    domain_response = agentcensus_get_json(
        f"domains/{agent['domain']}", token=AGENTCENSUS_BEARER_TOKEN
    )
    add_capture(f"trust_model_comparison_agentcensus_{agent['output_label']}_domain.json", domain_response)

for agent in selected_by_label.values():
    agent_key = agent["agentcensus_agent_key"]
    if not agent_key:
        continue
    mechanisms = (agent_responses[agent_key].get("data") or {}).get("mechanisms") or []
    for source in mechanisms:
        response = agentcensus_get_json(
            f"agents/{agent_key}/documents/{source}", token=AGENTCENSUS_BEARER_TOKEN
        )
        add_capture(
            f"trust_model_comparison_agentcensus_{agent['output_label']}_{source}_document.json",
            response,
        )

for agent in selected_by_label.values():
    if not agent["a2a_manifest_url"]:
        continue
    response = a2a_registry_request_json(
        "POST", "/public/tools/validate-url",
        body={"url": agent["a2a_manifest_url"]}, authenticated=False, timeout=45,
    )
    add_capture(
        f"trust_model_comparison_a2a_registry_{agent['output_label']}_integrity_validation.json",
        response,
    )

ans_identity_discovery = []
for agent in selected_by_label.values():
    for prefix in ("_ans-badge", "_ra-badge"):
        record_name = f"{prefix}.{agent['domain']}"
        request_url = f"{ANS_DNS_RESOLVER_URL}?{urlencode({'name': record_name, 'type': 'TXT'})}"
        with urlopen(request_url, timeout=30, context=ANS_TLS_CONTEXT) as response:
            body = response.read()
            ans_identity_discovery.append({
                "outputLabel": agent["output_label"], "domain": agent["domain"],
                "recordName": record_name, "url": request_url,
                "httpStatus": response.status, "data": json.loads(body),
            })
add_capture("trust_model_comparison_ans_identity_discovery.json", ans_identity_discovery)

## Capture the ANS Integrity demonstration and corroboration

In [6]:
ANS_INTEGRITY_EXAMPLES = [
    {
        "name": "Snitker ANS Interoperability Demo",
        "record_name": "_ans-badge.godaddy.demo.ans.snitker.dev",
        "dns_url": "https://dns.google/resolve?name=_ans-badge.godaddy.demo.ans.snitker.dev&type=TXT",
        "dns_file": "trust_model_comparison_ans_snitker_dns_badge.json",
        "badge_file": "trust_model_comparison_ans_snitker_transparency_log_badge.json",
        "metadata": {
            "A2A": {"url": "https://godaddy.demo.ans.snitker.dev/.well-known/agent-card.json", "file": "trust_model_comparison_ans_snitker_a2a_agent_card.json"},
            "MCP": {"url": "https://godaddy.demo.ans.snitker.dev/mcp/server-card", "file": "trust_model_comparison_ans_snitker_mcp_server_card.json"},
        },
    },
    {
        "name": "AgentCensus", "record_name": "_ans-badge.agentcensus.io",
        "dns_url": "https://dns.google/resolve?name=_ans-badge.agentcensus.io&type=TXT",
        "dns_file": "trust_model_comparison_ans_agentcensus_dns_badge.json",
        "badge_file": "trust_model_comparison_ans_agentcensus_transparency_log_badge.json",
        "metadata": {},
    },
]

for example in ANS_INTEGRITY_EXAMPLES:
    with urlopen(example["dns_url"], timeout=30, context=ANS_TLS_CONTEXT) as response:
        dns_body = response.read()
    dns_payload = json.loads(dns_body)
    add_capture(example["dns_file"], dns_payload, url=example["dns_url"], raw_body=dns_body)
    badge_value = next(answer["data"] for answer in dns_payload["Answer"] if answer["type"] == 16)
    badge_url = badge_value.split("url=", 1)[1]
    requests = [(badge_url, example["badge_file"])] + [
        (item["url"], item["file"]) for item in example["metadata"].values()
    ]
    for url, filename in requests:
        with urlopen(url, timeout=30, context=ANS_TLS_CONTEXT) as response:
            body = response.read()
        add_capture(filename, json.loads(body), url=url, raw_body=body)

AGENTCENSUS_ANS_EXAMPLES = [
    {
        "name": "Snitker ANS Interoperability Demo", "agent_key": "ag_62fad4c8c1d1",
        "output_label": "snitker",
        "documents": {"a2a_alt": "trust_model_comparison_agentcensus_ans_example_snitker_a2a_alt_document.json"},
        "agent_file": "trust_model_comparison_agentcensus_ans_example_snitker_agent.json",
        "ans_dns_file": "trust_model_comparison_ans_snitker_dns_badge.json",
        "ans_badge_file": "trust_model_comparison_ans_snitker_transparency_log_badge.json",
    },
    {
        "name": "AgentCensus", "agent_key": "ag_c449a3237ded", "output_label": "agentcensus",
        "documents": {
            "ans": "trust_model_comparison_agentcensus_ans_example_agentcensus_ans_document.json",
            "a2a": "trust_model_comparison_agentcensus_ans_example_agentcensus_a2a_document.json",
            "a2a_alt": "trust_model_comparison_agentcensus_ans_example_agentcensus_a2a_alt_document.json",
        },
        "agent_file": "trust_model_comparison_agentcensus_ans_example_agentcensus_agent.json",
        "ans_dns_file": "trust_model_comparison_ans_agentcensus_dns_badge.json",
        "ans_badge_file": "trust_model_comparison_ans_agentcensus_transparency_log_badge.json",
    },
]
for example in AGENTCENSUS_ANS_EXAMPLES:
    response = agentcensus_get_json(f"agents/{example['agent_key']}", token=AGENTCENSUS_BEARER_TOKEN)
    add_capture(example["agent_file"], response)
    for source, filename in example["documents"].items():
        response = agentcensus_get_json(
            f"agents/{example['agent_key']}/documents/{source}", token=AGENTCENSUS_BEARER_TOKEN
        )
        add_capture(filename, response)

evidence_bundle = {
    "schemaVersion": "1.0",
    "capturedAt": datetime.now(timezone.utc).isoformat(),
    "searchBundle": SEARCH_BUNDLE_PATH.name,
    "searchCapturedAt": search_bundle["capturedAt"],
    "analysisConfigurations": {
        "ansIntegrityExamples": ANS_INTEGRITY_EXAMPLES,
        "agentcensusAnsExamples": AGENTCENSUS_ANS_EXAMPLES,
    },
    "captures": captures,
}
write_json(EVIDENCE_BUNDLE_PATH, evidence_bundle)
print(f"{len(captures)} unique captures -> {EVIDENCE_BUNDLE_PATH.name}")

/Users/hanhanwu/Documents/Github/Agentopia/EXPERIMENTS/output/trust_model_evidence_bundle.json
26 unique captures -> trust_model_evidence_bundle.json


## Handoff

Run `trust_model_comparison.ipynb` after these two writes complete. That notebook
is intentionally read-only: it loads the bundles, maps each dimension to its
own search, and derives tables without making requests or emitting more JSON.